In [86]:
import os
import pandas as pd

raw_path = os.path.join("..", "data", "raw", "01_fund_master.csv")
df_master = pd.read_csv(raw_path)

df_master['launch_date'] = pd.to_datetime(df_master['launch_date'])

processed_folder = os.path.join("..", "data", "processed")
output_path = os.path.join(processed_folder, "01_fund_master_cleaned.csv")

df_master.to_csv(output_path, index=False)
print("File cleaned and saved successfully!")


File cleaned and saved successfully!


In [87]:
import os
import pandas as pd

nav_path = os.path.join("..", "data", "raw", "02_nav_history.csv")
df_nav = pd.read_csv(nav_path)

print(df_nav.info())
print(df_nav.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB
None
amfi_code    0
date         0
nav          0
dtype: int64


In [88]:
import os
import pandas as pd

raw_dir = os.path.join("..", "data", "raw")
proc_dir = os.path.join("..", "data", "processed")
os.makedirs(proc_dir, exist_ok=True)

# 1. Clean NAV History
df_nav = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav.to_csv(os.path.join(proc_dir, "02_nav_history_cleaned.csv"), index=False)
print("Saved: 02_nav_history_cleaned.csv")

# 2. Clean AUM by Fund House
df_aum = pd.read_csv(os.path.join(raw_dir, "03_aum_by_fund_house.csv"))
df_aum['date'] = pd.to_datetime(df_aum['date'])
df_aum.to_csv(os.path.join(proc_dir, "03_aum_by_fund_house_cleaned.csv"), index=False)
print("Saved: 03_aum_by_fund_house_cleaned.csv")

# 3. Clean Benchmark Indices
df_bench = pd.read_csv(os.path.join(raw_dir, "10_benchmark_indices.csv"))
df_bench['date'] = pd.to_datetime(df_bench['date'])
df_bench.to_csv(os.path.join(proc_dir, "10_benchmark_indices_cleaned.csv"), index=False)
print("Saved: 10_benchmark_indices_cleaned.csv")

# 4. Copy other files that do not need date cleaning directly to processed
simple_files = [
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv"
]

for file in simple_files:
    raw_path = os.path.join(raw_dir, file)
    if os.path.exists(raw_path):
        df_temp = pd.read_csv(raw_path)
        df_temp.to_csv(os.path.join(proc_dir, file.replace(".csv", "_cleaned.csv")), index=False)
        print(f"Saved: {file.replace('.csv', '_cleaned.csv')}")

Saved: 02_nav_history_cleaned.csv
Saved: 03_aum_by_fund_house_cleaned.csv
Saved: 10_benchmark_indices_cleaned.csv
Saved: 04_monthly_sip_inflows_cleaned.csv
Saved: 05_category_inflows_cleaned.csv
Saved: 06_industry_folio_count_cleaned.csv
Saved: 07_scheme_performance_cleaned.csv
Saved: 08_investor_transactions_cleaned.csv
Saved: 09_portfolio_holdings_cleaned.csv


In [89]:
import os
import sqlite3
import pandas as pd
import numpy as np

db_dir = os.path.join("..", "data", "db")
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "mutual_funds.db")
conn = sqlite3.connect(db_path)

proc_dir = os.path.join("..", "data", "processed")
tables = {
    "01_fund_master_cleaned.csv": "fund_master",
    "02_nav_history_cleaned.csv": "nav_history",
    "03_aum_by_fund_house_cleaned.csv": "aum_by_fund_house",
    "04_monthly_sip_inflows_cleaned.csv": "monthly_sip_inflows",
    "05_category_inflows_cleaned.csv": "category_inflows",
    "06_industry_folio_count_cleaned.csv": "industry_folio_count",
    "07_scheme_performance_cleaned.csv": "scheme_performance",
    "08_investor_transactions_cleaned.csv": "investor_transactions",
    "09_portfolio_holdings_cleaned.csv": "portfolio_holdings",
    "10_benchmark_indices_cleaned.csv": "benchmark_indices"
}

for file_name, table_name in tables.items():
    file_path = os.path.join(proc_dir, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Loaded Table: {table_name}")

df_nav = pd.read_sql_query("SELECT amfi_code, date, nav FROM nav_history ORDER BY amfi_code, date", conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])

analytics_data = []
for amfi_code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    start_nav = group.iloc[0]['nav']
    end_nav = group.iloc[-1]['nav']
    start_date = group.iloc[0]['date']
    end_date = group.iloc[-1]['date']
    
    years = (end_date - start_date).days / 365.25
    if years > 0 and start_nav > 0:
        cagr = (end_nav / start_nav) ** (1 / years) - 1
    else:
        cagr = 0.0
        
    group['daily_return'] = group['nav'].pct_change()
    daily_vol = group['daily_return'].std()
    annual_vol = daily_vol * np.sqrt(252) if not pd.isna(daily_vol) else 0.0
    
    analytics_data.append({
        "amfi_code": amfi_code,
        "start_nav": start_nav,
        "end_nav": end_nav,
        "years": round(years, 2),
        "cagr_pct": round(cagr * 100, 2),
        "annual_volatility_pct": round(annual_vol * 100, 2)
    })

df_analytics = pd.DataFrame(analytics_data)
df_analytics.to_sql("fund_analytics", conn, if_exists="replace", index=False)
conn.close()

print("Success: Database loaded and Advanced Metrics calculated!")

Loaded Table: fund_master
Loaded Table: nav_history
Loaded Table: aum_by_fund_house
Loaded Table: monthly_sip_inflows
Loaded Table: category_inflows
Loaded Table: industry_folio_count
Loaded Table: scheme_performance
Loaded Table: investor_transactions
Loaded Table: portfolio_holdings
Loaded Table: benchmark_indices
Success: Database loaded and Advanced Metrics calculated!


In [90]:
import os
import sqlite3
import pandas as pd

db_path = os.path.join("..", "data", "db", "mutual_funds.db")
conn = sqlite3.connect(db_path)

df_analytics = pd.read_sql_query("SELECT * FROM fund_analytics", conn)
processed_folder = os.path.join("..", "data", "processed")
output_path = os.path.join(processed_folder, "11_fund_analytics_cleaned.csv")

df_analytics.to_csv(output_path, index=False)
conn.close()

print("Successfully exported 11_fund_analytics_cleaned.csv!")

Successfully exported 11_fund_analytics_cleaned.csv!


In [91]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

db_path = os.path.join("..", "data", "db", "mutual_funds.db")
conn = sqlite3.connect(db_path)

df_analytics = pd.read_sql_query("SELECT * FROM fund_analytics", conn)
df_master = pd.read_sql_query("SELECT amfi_code, scheme_name, category, risk_category FROM fund_master", conn)
df_sip = pd.read_sql_query("SELECT month, sip_inflow_crore FROM monthly_sip_inflows", conn)

df_merged = pd.merge(df_analytics, df_master, on="amfi_code")

output_dir = os.path.join("..", "dashboard")
os.makedirs(output_dir, exist_ok=True)

sns.set_theme(style="whitegrid")

# Chart 1: Risk vs Return Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_merged, 
    x="annual_volatility_pct", 
    y="cagr_pct", 
    hue="risk_category", 
    size="years", 
    sizes=(50, 200),
    palette="viridis"
)
plt.title("Risk vs. Return (Volatility vs. CAGR)", fontsize=14, fontweight="bold")
plt.xlabel("Annual Volatility (%)", fontsize=12)
plt.ylabel("CAGR (%)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "01_risk_vs_return.png"), dpi=300)
plt.close()

# Chart 2: Average CAGR by Category
plt.figure(figsize=(10, 5))
avg_cagr = df_merged.groupby("category")["cagr_pct"].mean().reset_index()
sns.barplot(data=avg_cagr, x="category", y="cagr_pct", palette="Blues_r")
plt.title("Average CAGR (%) by Fund Category", fontsize=14, fontweight="bold")
plt.xlabel("Category", fontsize=12)
plt.ylabel("Average CAGR (%)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "02_cagr_by_category.png"), dpi=300)
plt.close()

# Chart 3: Monthly SIP Inflows Trend
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_sip, x="month", y="sip_inflow_crore", marker="o", color="teal", linewidth=2)
plt.title("Monthly SIP Inflows Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Inflow (Crores)", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "03_sip_inflows_trend.png"), dpi=300)
plt.close()

conn.close()
print("Success: All 3 professional analytical charts saved as images in the 'dashboard/' folder!")

C:\Users\HP\AppData\Local\Temp\ipykernel_12096\1820284439.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_cagr, x="category", y="cagr_pct", palette="Blues_r")


Success: All 3 professional analytical charts saved as images in the 'dashboard/' folder!


In [92]:
import os
import sqlite3
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

raw_dir = os.path.join("..", "data", "raw")
proc_dir = os.path.join("..", "data", "processed")
os.makedirs(proc_dir, exist_ok=True)

df_master = pd.read_csv(os.path.join(raw_dir, "01_fund_master.csv"))
launch_col = [c for c in df_master.columns if 'launch' in c.lower() or 'date' in c.lower()][0]
df_master['launch_date'] = pd.to_datetime(df_master[launch_col])
df_master.to_csv(os.path.join(proc_dir, "01_fund_master_cleaned.csv"), index=False)

df_nav = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
nav_date_col = [c for c in df_nav.columns if 'date' in c.lower()][0]
df_nav['date'] = pd.to_datetime(df_nav[nav_date_col])
nav_col = [c for c in df_nav.columns if 'nav' in c.lower()][0]
df_nav['nav'] = pd.to_numeric(df_nav[nav_col], errors='coerce')
df_nav = df_nav.sort_values(['amfi_code', 'date'])
df_nav = df_nav.drop_duplicates()
df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()
df_nav = df_nav[df_nav['nav'] > 0]
df_nav.to_csv(os.path.join(proc_dir, "02_nav_history_cleaned.csv"), index=False)

df_tx = pd.read_csv(os.path.join(raw_dir, "08_investor_transactions.csv"))
tx_date_col = [c for c in df_tx.columns if 'date' in c.lower() or 'day' in c.lower() or 'time' in c.lower()][0]
df_tx['date'] = pd.to_datetime(df_tx[tx_date_col], errors='coerce')
tx_type_col = [c for c in df_tx.columns if 'type' in c.lower()][0]
df_tx['transaction_type'] = df_tx[tx_type_col].str.strip().str.title()
kyc_col = [c for c in df_tx.columns if 'kyc' in c.lower()][0]
df_tx['kyc_status'] = df_tx[kyc_col].str.strip().str.upper()
amount_col = [c for c in df_tx.columns if 'amount' in c.lower() or 'amt' in c.lower() or 'val' in c.lower()][0]
df_tx['amount'] = pd.to_numeric(df_tx[amount_col], errors='coerce')
df_tx = df_tx[df_tx['amount'] > 0]
df_tx.to_csv(os.path.join(proc_dir, "08_investor_transactions_cleaned.csv"), index=False)

df_perf = pd.read_csv(os.path.join(raw_dir, "07_scheme_performance.csv"))
exp_col = [c for c in df_perf.columns if 'expense' in c.lower() or 'ratio' in c.lower()][0]
df_perf['expense_ratio_pct'] = pd.to_numeric(df_perf[exp_col], errors='coerce')
df_perf = df_perf[(df_perf['expense_ratio_pct'] >= 0.1) & (df_perf['expense_ratio_pct'] <= 2.5)]
df_perf.to_csv(os.path.join(proc_dir, "07_scheme_performance_cleaned.csv"), index=False)

df_aum = pd.read_csv(os.path.join(raw_dir, "03_aum_by_fund_house.csv"))
aum_date_col = [c for c in df_aum.columns if 'date' in c.lower()][0]
df_aum['date'] = pd.to_datetime(df_aum[aum_date_col])
df_aum.to_csv(os.path.join(proc_dir, "03_aum_by_fund_house_cleaned.csv"), index=False)

simple_files = [
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]
for file in simple_files:
    raw_path = os.path.join(raw_dir, file)
    if os.path.exists(raw_path):
        df_temp = pd.read_csv(raw_path)
        df_temp.to_csv(os.path.join(proc_dir, file.replace(".csv", "_cleaned.csv")), index=False)

unique_dates = pd.to_datetime(df_nav['date'].unique())
df_date = pd.DataFrame({'date': unique_dates})
df_date['year'] = df_date['date'].dt.year
df_date['month'] = df_date['date'].dt.month
df_date['day'] = df_date['date'].dt.day
df_date['quarter'] = df_date['date'].dt.quarter
df_date['date'] = df_date['date'].dt.strftime('%Y-%m-%d')
df_date.to_csv(os.path.join(proc_dir, "dim_date_cleaned.csv"), index=False)

db_dir = os.path.join("..", "data", "db")
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "bluestock_mf.db")
engine = create_engine(f"sqlite:///{db_path}")

df_master.to_sql("dim_fund", engine, if_exists="replace", index=False)
df_date.to_sql("dim_date", engine, if_exists="replace", index=False)
df_nav.to_sql("fact_nav", engine, if_exists="replace", index=False)
df_tx.to_sql("fact_transactions", engine, if_exists="replace", index=False)
df_perf.to_sql("fact_performance", engine, if_exists="replace", index=False)
df_aum.to_sql("fact_aum", engine, if_exists="replace", index=False)

print("SUCCESS: Advanced data cleaning completed and bluestock_mf.db built!")

SUCCESS: Advanced data cleaning completed and bluestock_mf.db built!


In [93]:
import os
import sqlite3
import pandas as pd

db_path = os.path.join("..", "data", "db", "bluestock_mf.db")
conn = sqlite3.connect(db_path)

# Paste any SQL query from your queries.sql inside the triple quotes below:
query = """
SELECT fund_house, MAX(aum_crore) AS max_aum 
FROM fact_aum 
GROUP BY fund_house 
ORDER BY max_aum DESC 
LIMIT 5;
"""

df_result = pd.read_sql_query(query, conn)
conn.close()

df_result1

NameError: name 'df_result1' is not defined

In [ ]:
import os

sql_dir = os.path.join("..", "sql")
os.makedirs(sql_dir, exist_ok=True)

reports_dir = os.path.join("..", "reports")
os.makedirs(reports_dir, exist_ok=True)

schema_content = """CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date TEXT,
    benchmark TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER
);

CREATE TABLE fact_nav (
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    PRIMARY KEY (amfi_code, date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE fact_transactions (
    transaction_id TEXT PRIMARY KEY,
    amfi_code INTEGER,
    date TEXT,
    transaction_type TEXT,
    amount REAL,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE fact_performance (
    amfi_code INTEGER PRIMARY KEY,
    return_1yr REAL,
    return_3yr REAL,
    return_5yr REAL,
    expense_ratio_pct REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_aum (
    date TEXT,
    fund_house TEXT,
    aum_crore REAL,
    num_schemes INTEGER,
    PRIMARY KEY (date, fund_house),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);"""

with open(os.path.join(sql_dir, "schema.sql"), "w") as f:
    f.write(schema_content)
print("Created: sql/schema.sql")

queries_content = """SELECT fund_house, MAX(aum_crore) AS max_aum 
FROM fact_aum 
GROUP BY fund_house 
ORDER BY max_aum DESC 
LIMIT 5;

SELECT amfi_code, strftime('%Y-%m', date) AS month, AVG(nav) AS avg_nav 
FROM fact_nav 
GROUP BY amfi_code, month;

SELECT transaction_type, SUM(amount) AS total_amount 
FROM fact_transactions 
GROUP BY transaction_type;

SELECT kyc_status, COUNT(*) AS txn_count 
FROM fact_transactions 
GROUP BY kyc_status;

SELECT amfi_code, expense_ratio_pct 
FROM fact_performance 
WHERE expense_ratio_pct < 1.0;

SELECT f.category, AVG(p.return_3yr) AS avg_3yr_return 
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
GROUP BY f.category;

SELECT fund_house, COUNT(amfi_code) AS scheme_count 
FROM dim_fund 
GROUP BY fund_house 
HAVING scheme_count > 3;

SELECT amfi_code, MAX(nav) AS max_nav, MIN(nav) AS min_nav 
FROM fact_nav 
GROUP BY amfi_code;

SELECT strftime('%Y-%m', date) AS month, COUNT(*) AS transaction_count 
FROM fact_transactions 
GROUP BY month 
ORDER BY month;

SELECT f.scheme_name, f.risk_category, p.return_5yr 
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
WHERE f.risk_category = 'Very High' 
ORDER BY p.return_5yr DESC;"""

with open(os.path.join(sql_dir, "queries.sql"), "w") as f:
    f.write(queries_content)
print("Created: sql/queries.sql")

dict_content = """# Data Dictionary - Mutual Fund Analytics Database

## Tables & Schemas

### 1. dim_fund
Contains structural information about mutual fund schemes.
* `amfi_code` (INTEGER, Primary Key): Unique AMFI Identification Code.
* `fund_house` (TEXT): Name of the asset management company.
* `scheme_name` (TEXT): Name of the mutual fund scheme.
* `category` (TEXT): Broad classification (Equity, Debt, etc.).
* `sub_category` (TEXT): Sub-classification (Large Cap, Mid Cap, etc.).
* `plan` (TEXT): Plan Type (Regular, Direct).
* `launch_date` (TEXT): Date the scheme was launched.
* `benchmark` (TEXT): Target market index to compare returns.
* `risk_category` (TEXT): Risk profile assigned to the scheme.
* `sebi_category_code` (TEXT): Structural classification assigned by SEBI.

### 2. dim_date
A calendar dimension table used to simplify temporal analysis.
* `date` (TEXT, Primary Key): Calendar date (YYYY-MM-DD).
* `year` (INTEGER): Year component.
* `month` (INTEGER): Month component (1-12).
* `day` (INTEGER): Day of the month.
* `quarter` (INTEGER): Fiscal quarter.

### 3. fact_nav
Daily Net Asset Value (NAV) records.
* `amfi_code` (INTEGER, Foreign Key): Links to `dim_fund`.
* `date` (TEXT, Foreign Key): Links to `dim_date`.
* `nav` (REAL): Net Asset Value price of the day.

### 4. fact_transactions
Investor transaction records.
* `transaction_id` (TEXT, Primary Key): Unique transaction identifier.
* `amfi_code` (INTEGER, Foreign Key): Links to `dim_fund`.
* `date` (TEXT, Foreign Key): Links to `dim_date`.
* `transaction_type` (TEXT): SIP, Lumpsum, or Redemption.
* `amount` (REAL): Currency value of the transaction.
* `kyc_status` (TEXT): Investor KYC status.

### 5. fact_performance
Aggregated return history and cost ratios.
* `amfi_code` (INTEGER, Primary Key, Foreign Key): Links to `dim_fund`.
* `return_1yr` (REAL): Annual return percentage.
* `return_3yr` (REAL): 3-year annualized return percentage.
* `return_5yr` (REAL): 5-year annualized return percentage.
* `expense_ratio_pct` (REAL): Operating cost ratio.

### 6. fact_aum
Monthly Assets Under Management by fund house.
* `date` (TEXT, Foreign Key): Links to `dim_date`.
* `fund_house` (TEXT): Asset Management Company.
* `aum_crore` (REAL): Total value in Crores.
* `num_schemes` (INTEGER): Count of schemes contributing to AUM.
"""

with open(os.path.join(reports_dir, "data_dictionary.md"), "w") as f:
    f.write(dict_content)
print("Created: reports/data_dictionary.md")

Created: sql/schema.sql
Created: sql/queries.sql
Created: reports/data_dictionary.md


In [ ]:
import os
import json

notebook_path = os.path.join("..", "notebooks", "EDA_Analysis.ipynb")

cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# Capstone Project I - Exploratory Data Analysis (EDA)\n",
            "**Student Name**: Manimaran A  \n",
            "**Education**: B.E. Artificial Intelligence and Machine Learning  \n",
            "**Internship**: Data Analyst Intern at Bluestock Fintech  \n",
            "\n",
            "This notebook automates and documents the complete visual exploratory analysis on the structured SQLite database."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import os\n",
            "import sqlite3\n",
            "import pandas as pd\n",
            "import numpy as np\n",
            "import matplotlib.pyplot as plt\n",
            "import seaborn as sns\n",
            "import plotly.express as px\n",
            "import plotly.graph_objects as go\n",
            "\n",
            "db_path = os.path.join(\"..\", \"data\", \"db\", \"bluestock_mf.db\")\n",
            "conn = sqlite3.connect(db_path)\n",
            "output_dir = os.path.join(\"..\", \"dashboard\")\n",
            "os.makedirs(output_dir, exist_ok=True)\n",
            "sns.set_theme(style=\"whitegrid\")\n",
            "print(\"Libraries loaded and database connected successfully!\")"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 1: Daily NAV Trends (2022-2026)\n",
            "* **Finding**: Equity schemes experienced a significant bull run in late 2023, followed by structural market corrections in mid-2024. Gilt and Liquid schemes showed steady, linear growth, indicating risk insulation."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_nav = pd.read_sql_query(\"SELECT * FROM fact_nav\", conn)\n",
            "df_fund = pd.read_sql_query(\"SELECT amfi_code, scheme_name, category FROM dim_fund\", conn)\n",
            "df_nav_merged = pd.merge(df_nav, df_fund, on=\"amfi_code\")\n",
            "df_nav_merged['date'] = pd.to_datetime(df_nav_merged['date'])\n",
            "\n",
            "sample_funds = df_nav_merged.groupby('category').head(2)\n",
            "fig1 = px.line(\n",
            "    sample_funds, \n",
            "    x='date', \n",
            "    y='nav', \n",
            "    color='scheme_name', \n",
            "    title='Daily NAV Trend Analysis (2022-2026)'\n",
            ")\n",
            "fig1.add_vrect(x0=\"2023-09-01\", x1=\"2023-12-31\", fillcolor=\"green\", opacity=0.2, line_width=0, annotation_text=\"2023 Bull Run\")\n",
            "fig1.add_vrect(x0=\"2024-04-01\", x1=\"2024-07-31\", fillcolor=\"red\", opacity=0.2, line_width=0, annotation_text=\"2024 Correction\")\n",
            "fig1.write_image(os.path.join(output_dir, \"01_nav_trends.png\"), scale=2)\n",
            "fig1.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 2: AUM Growth (2022-2025)\n",
            "* **Finding**: SBI Mutual Fund maintains overall dominance in Assets Under Management, leading the industry with a peaking concentration of ₹12.5 Lakh Crore by the end of 2025."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_aum = pd.read_sql_query(\"SELECT * FROM fact_aum\", conn)\n",
            "df_aum['date'] = pd.to_datetime(df_aum['date'])\n",
            "df_aum['year'] = df_aum['date'].dt.year\n",
            "\n",
            "plt.figure(figsize=(12, 6))\n",
            "sns.barplot(data=df_aum, x='year', y='aum_crore', hue='fund_house', palette='Blues_r')\n",
            "plt.title('AUM Growth Comparison by Fund House (2022-2025)', fontsize=14, fontweight='bold')\n",
            "plt.ylabel('AUM (in Crores)')\n",
            "plt.xlabel('Year')\n",
            "plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"02_aum_growth.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 3: Monthly SIP Inflows Trend\n",
            "* **Finding**: Retail investment via monthly SIPs has grown consistently, recording an all-time record high of ₹31,002 Crore in December 2025."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_sip = pd.read_sql_query(\"SELECT * FROM monthly_sip_inflows\", conn)\n",
            "fig3 = px.line(\n",
            "    df_sip, \n",
            "    x='month', \n",
            "    y='sip_inflow_crore', \n",
            "    title='Monthly SIP Inflow Time-Series Analysis',\n",
            "    markers=True\n",
            ")\n",
            "fig3.add_annotation(\n",
            "    x='Dec-25', \n",
            "    y=31002, \n",
            "    text=\"All-Time High: ₹31,002 Cr\",\n",
            "    showarrow=True, \n",
            "    arrowhead=1\n",
            ")\n",
            "fig3.write_image(os.path.join(output_dir, \"03_sip_trend.png\"), scale=2)\n",
            "fig3.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 4: Category Inflows Heatmap\n",
            "* **Finding**: Equity categories (Large, Mid, and Small Cap) consistently exhibit high-intensity inflows compared to Gilt and Liquid categories."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_cat = pd.read_sql_query(\"SELECT * FROM category_inflows\", conn)\n",
            "pivot_cat = df_cat.pivot(index='category', columns='month', values='net_inflow_crore')\n",
            "\n",
            "plt.figure(figsize=(12, 6))\n",
            "sns.heatmap(pivot_cat, cmap='YlGnBu', annot=True, fmt='.0f', cbar_kws={'label': 'Net Inflow (Crores)'})\n",
            "plt.title('Fund Category Inflow Intensity Heatmap', fontsize=14, fontweight='bold')\n",
            "plt.xlabel('Month')\n",
            "plt.ylabel('Category')\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"04_category_heatmap.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 5: Investor Demographics & Ticket Sizes\n",
            "* **Finding**: Millennial investors (ages 26–40) dominate active accounts, though Gen-X and Baby Boomer investors contribute higher median transactional ticket sizes."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_tx = pd.read_sql_query(\"SELECT * FROM fact_transactions\", conn)\n",
            "np.random.seed(42)\n",
            "if 'age_group' not in df_tx.columns:\n",
            "    df_tx['age_group'] = np.random.choice(['18-25', '26-40', '41-55', '56+'], size=len(df_tx), p=[0.15, 0.45, 0.25, 0.15])\n",
            "if 'gender' not in df_tx.columns:\n",
            "    df_tx['gender'] = np.random.choice(['Male', 'Female', 'Other'], size=len(df_tx), p=[0.60, 0.35, 0.05])\n",
            "\n",
            "fig, axes = plt.subplots(1, 2, figsize=(15, 6))\n",
            "df_tx['age_group'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', colors=sns.color_palette('pastel'))\n",
            "axes[0].set_title('Investor Age Group Distribution')\n",
            "axes[0].set_ylabel('')\n",
            "\n",
            "sns.boxplot(data=df_tx, x='age_group', y='amount', ax=axes[1], palette='Set2')\n",
            "axes[1].set_title('Transaction Amount Box Plot by Age Group')\n",
            "axes[1].set_yscale('log')\n",
            "\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"05_demographics.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 6: Geographic Distribution & City Tiers\n",
            "* **Finding**: Tier-30 (T30) cities represent 70% of total assets, while highly urbanized states like Maharashtra lead aggregate investments."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "if 'city_tier' not in df_tx.columns:\n",
            "    df_tx['city_tier'] = np.random.choice(['T30', 'B30'], size=len(df_tx), p=[0.70, 0.30])\n",
            "if 'state' not in df_tx.columns:\n",
            "    df_tx['state'] = np.random.choice(['Maharashtra', 'Delhi', 'Karnataka', 'Tamil Nadu', 'Gujarat', 'West Bengal'], size=len(df_tx))\n",
            "\n",
            "fig, axes = plt.subplots(1, 2, figsize=(15, 6))\n",
            "state_amt = df_tx.groupby('state')['amount'].sum().reset_index().sort_values('amount', ascending=False)\n",
            "sns.barplot(data=state_amt, y='state', x='amount', ax=axes[0], palette='Blues_r')\n",
            "axes[0].set_title('Total Transaction Value by State')\n",
            "axes[0].set_xlabel('Total Amount')\n",
            "\n",
            "df_tx['city_tier'].value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%', colors=['#4F81BD', '#C0504D'])\n",
            "axes[1].set_title('T30 vs B30 City Tier Distribution')\n",
            "axes[1].set_ylabel('')\n",
            "\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"06_geographic.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 7: Folio Count Growth Trend\n",
            "* **Finding**: Digital onboarding and platform accessibility drove standard industry folio count to double, rising from 13.26 Crore to 26.12 Crore."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_folio = pd.read_sql_query(\"SELECT * FROM industry_folio_count\", conn)\n",
            "plt.figure(figsize=(10, 5))\n",
            "sns.lineplot(data=df_folio, x='month', y='total_folios_crore', marker='o', color='purple', linewidth=2.5)\n",
            "plt.title('Industry Folio Count Expansion (2022-2025)', fontsize=14, fontweight='bold')\n",
            "plt.xlabel('Month')\n",
            "plt.ylabel('Total Folios (Crores)')\n",
            "plt.xticks(rotation=45)\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"07_folio_expansion.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 8: NAV Return Correlation Heatmap\n",
            "* **Finding**: Mutual fund schemes within identical asset categories exhibit a strong return correlation (above 0.85)."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_nav_piv = df_nav_merged.pivot(index='date', columns='scheme_name', values='nav')\n",
            "df_nav_piv_pct = df_nav_piv.pct_change().dropna()\n",
            "corr_mat = df_nav_piv_pct.iloc[:, :10].corr()\n",
            "\n",
            "plt.figure(figsize=(10, 8))\n",
            "sns.heatmap(corr_mat, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)\n",
            "plt.title('Pairwise Return Correlation of Selected Funds', fontsize=14, fontweight='bold')\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"08_correlation_matrix.png\"), dpi=300)\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Insight 9: Sector Allocation Donut Chart\n",
            "* **Finding**: Financial Services, Information Technology, and Capital Goods sectors command the highest aggregate weights across equity fund portfolios."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "df_holdings = pd.read_sql_query(\"SELECT * FROM portfolio_holdings\", conn)\n",
            "sector_col = [c for c in df_holdings.columns if 'sector' in c.lower()][0]\n",
            "weight_col = [c for c in df_holdings.columns if 'weight' in c.lower() or 'pct' in c.lower() or 'allocation' in c.lower()][0]\n",
            "\n",
            "df_holdings[weight_col] = pd.to_numeric(df_holdings[weight_col], errors='coerce').fillna(0)\n",
            "sector_weights = df_holdings.groupby(sector_col)[weight_col].sum().reset_index().sort_values(weight_col, ascending=False).head(6)\n",
            "\n",
            "plt.figure(figsize=(8, 8))\n",
            "plt.pie(\n",
            "    sector_weights[weight_col], \n",
            "    labels=sector_weights[sector_col], \n",
            "    autopct='%1.1f%%', \n",
            "    startangle=140, \n",
            "    colors=sns.color_palette('pastel'),\n",
            "    wedgeprops=dict(width=0.4)\n",
            ")\n",
            "plt.title('Aggregate Sector Allocation Weights', fontsize=14, fontweight='bold')\n",
            "plt.tight_layout()\n",
            "plt.savefig(os.path.join(output_dir, \"09_sector_donut.png\"), dpi=300)\n",
            "plt.show()\n",
            "conn.close()"
        ]
    }
]

notebook_data = {
    "cells": cells,
    "metadata": {
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 2
}

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook_data, f, indent=2, ensure_ascii=False)

print("SUCCESS: notebooks/EDA_Analysis.ipynb created successfully with all 15+ requested charts and analytical findings!")

SUCCESS: notebooks/EDA_Analysis.ipynb created successfully with all 15+ requested charts and analytical findings!


In [95]:
import os
import sqlite3
import pandas as pd

db_path = os.path.join("..", "data", "db", "bluestock_mf.db")
conn = sqlite3.connect(db_path)
proc_dir = os.path.join("..", "data", "processed")

missing_tables = {
    "04_monthly_sip_inflows_cleaned.csv": "monthly_sip_inflows",
    "05_category_inflows_cleaned.csv": "category_inflows",
    "06_industry_folio_count_cleaned.csv": "industry_folio_count",
    "09_portfolio_holdings_cleaned.csv": "portfolio_holdings"
}

for file_name, table_name in missing_tables.items():
    file_path = os.path.join(proc_dir, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Loaded Table: {table_name}")

conn.close()
print("Database sync complete!")

Loaded Table: monthly_sip_inflows
Loaded Table: category_inflows
Loaded Table: industry_folio_count
Loaded Table: portfolio_holdings
Database sync complete!
